In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from itertools import combinations
from sklearn.linear_model import LinearRegression


from datetime import datetime, timedelta
import re


In [0]:
url = "https://ldcom365.sharepoint.com"

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

data_full_feed=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data.xlsx')

#https://ldcom365.sharepoint.com/:x:/r/sites/grainsargprojects/models/consumption/data.xlsx?d=wbdbdefa4f42f4c94865d6f16a98d46a4&csf=1&web=1&e=qCDJcz

## CORRELATION MATRIX

In [0]:

data_full_feed=data_full_feed[data_full_feed['Year']>=2005]
data_full_feed = data_full_feed.apply(pd.to_numeric, errors='coerce')

data_full_feed_y=data_full_feed[['CORN SHARE']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show

### 1st model using every feature available

In [0]:
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

print("Dropping the following highly correlated columns:")
print(to_drop)

# Drop them
data_reduced = data_full_feed_X.drop(columns=to_drop)
X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

rf_model=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

y_pred = rf_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

plt.scatter(y_test, y_pred)
plt.xlabel("Actual Corn Use")
plt.ylabel("Predicted Corn Use")
plt.title("Actual vs Predicted Corn Use")
plt.grid(True)
plt.show()

# 8. Optional: Feature importance
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', title='Feature Importance')
plt.show()

### 1 Model for each combination possible

In [0]:
data_full_feed=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data - pred.xlsx')

data_full_feed=data_full_feed[data_full_feed['Year']>=2005]
data_full_feed = data_full_feed.apply(pd.to_numeric, errors='coerce')

data_full_feed_y=data_full_feed[['Corn use']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_X.drop(columns=to_drop)

X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()




# === Separate 2025 ===
row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()

# === Remove 2025 from training data ===
X_cleaned = X[data_full_feed['Year'] != 2025]
y_cleaned = y[data_full_feed['Year'] != 2025]

# Set up
features = list(X.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred
        })

# Create DataFrame
results_df_rf = pd.DataFrame(results_with_2025)

# Sort by R² descending
results_df_rf = results_df_rf.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10
print(results_df_rf.head(10))

# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/Results_RF/feature_combinations_rf_corn_use_domcons.xlsx', results_df_rf, index=False)


## CORN

### DOM CONS

In [0]:
data_full_feed=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data - tot dom cons.xlsx')
history=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/pred history.xlsx')

data_full_feed=data_full_feed[data_full_feed['Year']>=2005]
data_full_feed = data_full_feed.apply(pd.to_numeric, errors='coerce')

data_full_feed_y=data_full_feed[['Corn use']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_X.drop(columns=to_drop)

X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()




# === Separate 2025 ===
row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()

# === Remove 2025 from training data ===
X_cleaned = X[data_full_feed['Year'] != 2025]
y_cleaned = y[data_full_feed['Year'] != 2025]

# Set up
features = list(X.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred[0]
        })

# Create DataFrame
results_df_rf = pd.DataFrame(results_with_2025)

# Sort by R² descending
results_df_rf = results_df_rf.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10
print(results_df_rf.head(10))

# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/Results_REG/feature_combinations_reg_corn_use_.xlsx', results_df_rf, index=False)

today_str = datetime.today().strftime('%Y-%m-%d')

# Step 2: Best model (highest R²)
best_row = results_df_rf.iloc[0]

# Step 3: Extract values
best_features = best_row['features']
best_prediction = best_row['Corn Share 2025 Prediction']
best_mse = best_row['MSE']
best_r2 = best_row['R2']

# Step 4: Get 2025 values for the used features
feature_values_2025 = row_2025_X[best_features].values.flatten().tolist()

# Step 5: Build the new row
new_row = pd.DataFrame([{
    'date': today_str,
    'data': 'Dom cons',
    'model': 'Regression',
    'MSE': best_mse,
    'R2': best_r2,
    'Y': 'Corn use',
    'prediction': best_prediction,
    'features': best_features,
    'values': feature_values_2025
}])

# Step 6: Append to history
history = pd.concat([history, new_row], ignore_index=True)



data_full_feed_y=data_full_feed[['Sorghum use']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_X.drop(columns=to_drop)

X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()




# === Separate 2025 ===
row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()

# === Remove 2025 from training data ===
X_cleaned = X[data_full_feed['Year'] != 2025]
y_cleaned = y[data_full_feed['Year'] != 2025]

# Set up
features = list(X.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred[0]
        })

# Create DataFrame
results_df_rf = pd.DataFrame(results_with_2025)

# Sort by R² descending
results_df_rf = results_df_rf.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10
print(results_df_rf.head(10))

# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/Results_REG/feature_combinations_reg_sorghum_use_dom_cons.xlsx', results_df_rf, index=False)

today_str = datetime.today().strftime('%Y-%m-%d')

# Step 2: Best model (highest R²)
best_row = results_df_rf.iloc[0]

# Step 3: Extract values
best_features = best_row['features']
best_prediction = best_row['Sorghum Share 2025 Prediction']
best_mse = best_row['MSE']
best_r2 = best_row['R2']

# Step 4: Get 2025 values for the used features
feature_values_2025 = row_2025_X[best_features].values.flatten().tolist()

# Step 5: Build the new row
new_row = pd.DataFrame([{
    'date': today_str,
    'data':'Dom cons',
    'model': 'Regression',
    'MSE': best_mse,
    'R2': best_r2,
    'Y': 'Sorghum use',
    'prediction': best_prediction,
    'features': best_features,
    'values': feature_values_2025
}])

# Step 6: Append to history
history = pd.concat([history, new_row], ignore_index=True)



### FEED

In [0]:
data_full_feed=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data - pred.xlsx')

data_full_feed_y=data_full_feed[['Corn use']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_X.drop(columns=to_drop)

X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()




# === Separate 2025 ===
row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()

# === Remove 2025 from training data ===
X_cleaned = X[data_full_feed['Year'] != 2025]
y_cleaned = y[data_full_feed['Year'] != 2025]

# Set up
features = list(X.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred[0]
        })

# Create DataFrame
results_df_rf = pd.DataFrame(results_with_2025)

# Sort by R² descending
results_df_rf = results_df_rf.sort_values(by='R2', ascending=False).reset_index(drop=True)


# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/Results_REG/feature_combinations_reg_corn_use.xlsx', results_df_rf, index=False)

today_str = datetime.today().strftime('%Y-%m-%d')

# Step 2: Best model (highest R²)
best_row = results_df_rf.iloc[0]

# Step 3: Extract values
best_features = best_row['features']
best_prediction = best_row['Corn Share 2025 Prediction']
best_mse = best_row['MSE']
best_r2 = best_row['R2']

# Step 4: Get 2025 values for the used features
feature_values_2025 = row_2025_X[best_features].values.flatten().tolist()

# Step 5: Build the new row
new_row = pd.DataFrame([{
    'date': today_str,
    'data':'Feed',
    'model': 'Regression',
    'MSE': best_mse,
    'R2': best_r2,
    'Y': 'Corn use',
    'prediction': best_prediction,
    'features': best_features,
    'values': feature_values_2025
}])

# Step 6: Append to history
history = pd.concat([history, new_row], ignore_index=True)

data_full_feed_y=data_full_feed[['Sorghum use']]
data_full_feed_X=data_full_feed[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix = data_full_feed_X.corr()
upper = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_X.drop(columns=to_drop)

X=data_full_feed_X
y=data_full_feed_y

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()




# === Separate 2025 ===
row_2025 = data_full_feed[data_full_feed['Year'] == 2025]
row_2025_X = row_2025[X.columns].copy()

# === Remove 2025 from training data ===
X_cleaned = X[data_full_feed['Year'] != 2025]
y_cleaned = y[data_full_feed['Year'] != 2025]

# Set up
features = list(X.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred[0]
        })

# Create DataFrame
results_df_rf = pd.DataFrame(results_with_2025)

# Sort by R² descending
results_df_rf = results_df_rf.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10
print(results_df_rf.head(10))

# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/Results_REG/feature_combinations_reg_sorghum_use.xlsx', results_df_rf, index=False)

today_str = datetime.today().strftime('%Y-%m-%d')

# Step 2: Best model (highest R²)
best_row = results_df_rf.iloc[0]

# Step 3: Extract values
best_features = best_row['features']
best_prediction = best_row['Sorghum Share 2025 Prediction']
best_mse = best_row['MSE']
best_r2 = best_row['R2']

# Step 4: Get 2025 values for the used features
feature_values_2025 = row_2025_X[best_features].values.flatten().tolist()

# Step 5: Build the new row
new_row = pd.DataFrame([{
    'date': today_str,
    'data': 'Feed',
    'model': 'Regression',
    'MSE': best_mse,
    'R2': best_r2,
    'Y': 'Sorghum use',
    'prediction': best_prediction,
    'features': best_features,
    'values': feature_values_2025
}])

# Step 6: Append to history
history = pd.concat([history, new_row], ignore_index=True)

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/pred history.xlsx', history, index=False)



### FEED

In [0]:
data_full_feed=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data - tot pred.xlsx')
history=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/pred history.xlsx',sheet_name='sorghum_feed')

data_full_feed=data_full_feed[data_full_feed['Year']>=2005]
data_full_feed = data_full_feed.apply(pd.to_numeric, errors='coerce')



### WITH DATA FROM CATTLE

#### RF

In [0]:
data_full_feed_cattle=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data cattle.xlsx')


data_full_feed_cattle=data_full_feed_cattle[data_full_feed_cattle['Year']>=2005]
data_full_feed_cattle = data_full_feed_cattle.apply(pd.to_numeric, errors='coerce')

data_full_feed_cattle_y=data_full_feed_cattle[['CORN SHARE']]
data_full_feed_cattle_X=data_full_feed_cattle[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix_cattle = data_full_feed_cattle_X.corr()


upper = correlation_matrix_cattle.where(
    np.triu(np.ones(correlation_matrix_cattle.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

print("Dropping the following highly correlated columns:")
print(to_drop)

# Drop them
data_reduced = data_full_feed_cattle_X.drop(columns=to_drop)

X_cattle=data_full_feed_cattle_X
y_cattle=data_full_feed_cattle_y

X_train_cattle,X_test_cattle,y_train_cattle,y_test_cattle=train_test_split(X_cattle,y_cattle,test_size=0.2,random_state=42)

rf_model_cattle=RandomForestRegressor(n_estimators=100,random_state=42)
rf_model_cattle.fit(X_train_cattle,y_train_cattle)

y_pred_cattle = rf_model_cattle.predict(X_test_cattle)
print("MSE:", mean_squared_error(y_test_cattle, y_pred_cattle))
print("R² Score:", r2_score(y_test_cattle, y_pred_cattle))

plt.scatter(y_test_cattle, y_pred_cattle)
plt.xlabel("Actual Corn Use")
plt.ylabel("Predicted Corn Use")
plt.title("Actual vs Predicted Corn Use")
plt.grid(True)
plt.show()

# 8. Optional: Feature importance
importances = pd.Series(rf_model_cattle.feature_importances_, index=X_cattle.columns)
importances.sort_values().plot(kind='barh', title='Feature Importance')
plt.show()


from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from itertools import combinations

# Set up
features = list(X_cattle.columns)
results = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        X_subset = X_cattle[list(subset)]
        
        # Split data
        X_train_cattle, X_test_cattle, y_train_cattle, y_test_cattle = train_test_split(X_subset, y_cattle, test_size=0.2, random_state=42)
        
        # Train model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train_cattle, y_train_cattle)
        y_pred_cattle = model.predict(X_test_cattle)
        
        # Evaluate
        mse = mean_squared_error(y_test_cattle, y_pred_cattle)
        r2 = r2_score(y_test_cattle, y_pred_cattle)
        
        # Store results
        results.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2
        })

# Create DataFrame
results_df_cattle = pd.DataFrame(results)

# Sort by R² descending
results_df_cattle = results_df_cattle.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10 combinations
print(results_df_cattle.head(10))


In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_cattle.xlsx',results_df_cattle,index=False)

#### PORKS

In [0]:
data_full_feed_pork=sp_mgr.read_pd_from_excel('/sites/grainsargprojects/models/consumption/data pork.xlsx')

data_full_feed_pork=data_full_feed_pork[data_full_feed_pork['Year']>=2005]
data_full_feed_pork = data_full_feed_pork.apply(pd.to_numeric, errors='coerce')

data_full_feed_pork_y=data_full_feed_pork[['CORN SHARE']]
data_full_feed_pork_X=data_full_feed_pork[['corn price', 'Sorghum price', 'C/S',
       'Corn Supply', 'Sorghum Supply', 'Prod ratio', 'CBOT corn T',
       'corn Supply/Total feed', 'sorghum Supply/Total feed',
       'corn price/supply', 'sorghum price/supply']]

correlation_matrix_pork = data_full_feed_pork_X.corr()
upper = correlation_matrix_pork.where(
    np.triu(np.ones(correlation_matrix_pork.shape), k=1).astype(bool)
)

# Find columns with correlation > 0.9
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

data_reduced = data_full_feed_pork_X.drop(columns=to_drop)

X_pork=data_full_feed_pork_X
y_pork=data_full_feed_pork_y

X_train_pork,X_test_pork,y_train_pork,y_test_pork=train_test_split(X_pork,y_pork,test_size=0.2,random_state=42)

row_2025 = data_full_feed_pork[data_full_feed_pork['Year'] == 2025]
row_2025_X = row_2025[X_pork.columns].copy()

# === Remove 2025 from training set ===
X_pork_cleaned = X_pork[data_full_feed_pork['Year'] != 2025]
y_pork_cleaned = y_pork[data_full_feed_pork['Year'] != 2025]

# === Try all non-empty feature combinations and predict 2025 ===
features = list(X_pork.columns)
results_with_2025 = []

for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_pork_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split cleaned data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y_pork_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_train, y_train)

        # Evaluate on test set
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Predict for 2025 row
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan  # in case of NaN or missing values

        # Store result
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred
        })

# Create and sort the result DataFrame
results_df_pork = pd.DataFrame(results_with_2025)
results_df_pork = results_df_pork.sort_values(by='R2', ascending=False).reset_index(drop=True)

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_rf_pork_pred.xlsx',results_df_pork,index=False)

In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_rf_pork.xlsx',results_df_pork,index=False)

#### LINEAR REGRESSIONS

In [0]:
# Set up
features = list(X_pork.columns)
results = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        X_subset_pork = X_pork[list(subset)]

        # Split data
        X_train_pork, X_test_pork, y_train_pork, y_test_pork = train_test_split(X_subset_pork, y_pork, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train_pork, y_train_pork)
        y_pred_pork = model.predict(X_test_pork)

        # Evaluate
        mse = mean_squared_error(y_test_pork, y_pred_pork)
        r2 = r2_score(y_test_pork, y_pred_pork)

        # Store results
        results.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2
        })

# Create DataFrame
results_lin_pork= pd.DataFrame(results)

# Sort by R² descending
results_lin_pork = results_lin_pork.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10 combinations
print(results_lin_pork.head(10))

sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_reg_pork.xlsx',results_lin_pork,index=False)

In [0]:
# === Separate 2025 row ===
row_2025 = data_full_feed_pork[data_full_feed_pork['Year'] == 2025]
row_2025_X = row_2025[X_pork.columns].copy()

# === Remove 2025 from full dataset ===
X_pork_cleaned = X_pork[data_full_feed_pork['Year'] != 2025]
y_pork_cleaned = y_pork[data_full_feed_pork['Year'] != 2025]

# Set up
features = list(X_pork.columns)
results_with_2025 = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        subset = list(subset)
        X_subset = X_pork_cleaned[subset]
        row_2025_subset = row_2025_X[subset]

        # Split training data
        X_train_pork, X_test_pork, y_train_pork, y_test_pork = train_test_split(X_subset, y_pork_cleaned, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train_pork, y_train_pork)
        y_pred_pork = model.predict(X_test_pork)

        # Evaluate
        mse = mean_squared_error(y_test_pork, y_pred_pork)
        r2 = r2_score(y_test_pork, y_pred_pork)

        # Predict 2025 value
        try:
            y_2025_pred = model.predict(row_2025_subset)[0]
        except:
            y_2025_pred = np.nan

        # Store results
        results_with_2025.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2,
            'Corn Share 2025 Prediction': y_2025_pred[0]
        })

# Create results DataFrame
results_lin_pork = pd.DataFrame(results_with_2025)
results_lin_pork = results_lin_pork.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10

# Export
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_reg_pork_pred.xlsx', results_lin_pork, index=False)

### SORGHUM MILLING MODEL

In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from itertools import combinations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set up
features = list(X.columns)
results = []

# Try all non-empty feature combinations
for r in range(1, len(features) + 1):
    for subset in combinations(features, r):
        X_subset = X[list(subset)]

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)

        # Train model
        model = LinearRegression()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Evaluate
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Store results
        results.append({
            'features': subset,
            'n_features': len(subset),
            'MSE': mse,
            'R2': r2
        })

# Create DataFrame
results_df = pd.DataFrame(results)

# Sort by R² descending
results_df = results_df.sort_values(by='R2', ascending=False).reset_index(drop=True)

# Show top 10 combinations
print(results_df.head(10))


In [0]:
sp_mgr.save_pd_to_excel('/sites/grainsargprojects/models/consumption/feature_combinations_Reg.xlsx',results_df,index=False)